# 03 — Detecção de regimes via HMM (Semana 4 do PLANO.md)

Especificação do pré-projeto (seção 3.3): HMM gaussiano (covariância cheia)
sobre observações multivariadas — **retorno agregado da carteira +
volatilidade realizada** —, 2-4 estados via BIC/AIC + inspeção econômica,
Baum-Welch com múltiplas inicializações, Viterbi (caracterização in-sample),
Forward (inferência prospectiva, sem lookahead), robustez via bootstrap em
blocos.

**Este é o fit único/exploratório da Semana 4** (in-sample, para
caracterizar regimes) — o refit causal walk-forward de verdade, que gera a
feature que efetivamente alimenta o LSTM sem vazar informação futura para
os parâmetros do modelo, é escopo da Semana 6.

Duas especificações de observação, ambas do retorno agregado equal-weighted
(já convertido para BRL — ver notebook 02):
- **Principal (3D)**: `return_1d`, `realized_vol_21d`, `realized_vol_63d`
- **Robustez/parcimoniosa (2D)**: `return_1d`, `realized_vol_21d`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.hmm import (
    block_bootstrap_stability,
    characterize_regimes,
    fit_gaussian_hmm,
    forward_filter,
    select_n_states,
    viterbi_decode,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"

portfolio = pd.read_parquet(PROCESSED_DIR / "portfolio_return.parquet").dropna()
portfolio.tail()

,return_1d,realized_vol_21d,realized_vol_63d
2024-12-20,-0.003184,0.181554,0.143648
2024-12-23,-0.016564,0.181526,0.147055
2024-12-26,0.006352,0.181969,0.147006
2024-12-27,-0.011858,0.188730,0.148938
2024-12-30,0.003232,0.185798,0.148950


## Especificação principal (3D): retorno + vol. realizada 21d + 63d

In [2]:
cols_3d = ["return_1d", "realized_vol_21d", "realized_vol_63d"]
obs_3d = portfolio[cols_3d].values

selection_3d = select_n_states(obs_3d, state_range=(2, 3, 4), n_init=20, random_state=42)
selection_3d

Model is not converging.  Current: 31618.85259435839 is not greater than 31618.88344903241. Delta is -0.030854674019792583


Model is not converging.  Current: 31615.63069778799 is not greater than 31615.64992612289. Delta is -0.019228334898798494


Model is not converging.  Current: 31613.678506745924 is not greater than 31613.696645578355. Delta is -0.018138832430850016


Model is not converging.  Current: 31618.851695493628 is not greater than 31618.88358965916. Delta is -0.031894165531412


,log_likelihood,bic,aic
n_states,,,
2,29499.426831,-58823.292163,-58956.853663
3,31762.493043,-63232.383585,-63454.986085
4,33499.949137,-66573.534632,-66897.898275


In [3]:
n_states_3d = selection_3d["bic"].idxmin()
print(f"BIC seleciona {n_states_3d} estados (AIC sugere {selection_3d['aic'].idxmin()})")

model_3d = fit_gaussian_hmm(obs_3d, n_states=n_states_3d, n_init=30, random_state=42)
regime_summary_3d = characterize_regimes(model_3d, obs_3d, cols_3d)
regime_summary_3d

BIC seleciona 4 estados (AIC sugere 4)


,persistence,expected_duration_days,mean_return_1d,mean_realized_vol_21d,mean_realized_vol_63d
state_rank,,,,,
0,0.990706,107.596267,0.000448,0.140963,0.142109
1,0.980113,50.283179,0.000528,0.173872,0.177301
2,0.980971,52.552677,0.000398,0.219761,0.226747
3,0.974616,39.394549,-0.001898,0.451163,0.472717


## Viterbi (in-sample) vs. Forward (prospectivo, sem lookahead)

In [4]:
viterbi_states = viterbi_decode(model_3d, obs_3d)
forward_probs = forward_filter(model_3d, obs_3d)
forward_state = forward_probs.argmax(axis=1)

regimes = pd.DataFrame(
    {
        "viterbi_state": viterbi_states,
        "forward_state": forward_state,
        **{f"forward_prob_state_{s}": forward_probs[:, s] for s in range(n_states_3d)},
    },
    index=portfolio.index,
)

agreement = (regimes["viterbi_state"] == regimes["forward_state"]).mean()
print(f"Viterbi (in-sample) concorda com Forward (prospectivo) em {agreement:.1%} dos dias")
print("(divergências esperadas nas transições de regime — Viterbi vê o futuro próximo, Forward não)")

regimes.to_parquet(PROCESSED_DIR / "hmm_regimes_insample.parquet")
regimes.tail()

Viterbi (in-sample) concorda com Forward (prospectivo) em 96.4% dos dias
(divergências esperadas nas transições de regime — Viterbi vê o futuro próximo, Forward não)


C:\Python\hmm_lstm_TCC\src\models\hmm.py:138: RuntimeWarning: divide by zero encountered in log
  log_startprob = np.log(model.startprob_)


,viterbi_state,forward_state,forward_prob_state_0,forward_prob_state_1,forward_prob_state_2,forward_prob_state_3
2024-12-20,0,0,0.999948,0.000052,3.122616e-09,4.086124e-07
2024-12-23,0,0,0.999697,0.000301,8.297021e-09,1.889894e-06
2024-12-26,0,0,0.999883,0.000116,8.159849e-09,3.416110e-07
2024-12-27,0,0,0.999507,0.000491,1.592277e-08,1.572150e-06
2024-12-30,0,0,0.999751,0.000249,1.869322e-08,4.238246e-07


## Robustez: bootstrap em blocos

Avalia estabilidade do nº de estados escolhido por BIC e da sequência de
regimes (Viterbi), com estados alinhados por volatilidade entre reamostragens
e comparados via % de concordância + Adjusted Rand Index.

In [5]:
stability_3d = block_bootstrap_stability(
    obs_3d, state_range=(2, 3, 4), block_length=21, n_bootstrap=50, n_init=5, random_state=42
)
stability_3d

Model is not converging.  Current: 31618.852594358566 is not greater than 31618.88344903202. Delta is -0.03085467345226789


Model is not converging.  Current: 31174.49937679163 is not greater than 31174.499844063834. Delta is -0.0004672722025134135


Model is not converging.  Current: 33092.29525219801 is not greater than 33092.35225959899. Delta is -0.05700740098109236


Model is not converging.  Current: 31599.550637937664 is not greater than 31599.576451472512. Delta is -0.025813534848566633


Model is not converging.  Current: 31599.54824063939 is not greater than 31599.57070291928. Delta is -0.022462279892351944


...

Model is not converging.  Current: 30307.154612592913 is not greater than 30307.169563867945. Delta is -0.014951275032217382


Model is not converging.  Current: 30307.05126416474 is not greater than 30307.05189689972. Delta is -0.0006327349801722448


Model is not converging.  Current: 32823.57669089482 is not greater than 32823.588470077884. Delta is -0.011779183063481469


Model is not converging.  Current: 33182.53821180418 is not greater than 33182.54649243585. Delta is -0.00828063167136861


{'reference_n_states': 4,
 'selected_n_states_distribution': 3    0.02
 4    0.98
 Name: proportion, dtype: float64,
 'regime_agreement_pct_mean': 0.7538006261752459,
 'regime_agreement_pct_std': 0.1975994230284057,
 'adjusted_rand_index_mean': 0.6770936933107435,
 'adjusted_rand_index_std': 0.19848803549340882,
 'n_bootstrap_completed': 50}

## Especificação de robustez (2D): só retorno + vol. realizada 21d

Versão mais parcimoniosa — `realized_vol_21d` e `realized_vol_63d` são
fortemente correlacionadas (Semana 3), então o modelo 3D pode estar
efetivamente perto de um 2D com peso redobrado na vol. de curto prazo.
Comparar os regimes das duas especificações é um teste de robustez barato.

In [6]:
cols_2d = ["return_1d", "realized_vol_21d"]
obs_2d = portfolio[cols_2d].values

selection_2d = select_n_states(obs_2d, state_range=(2, 3, 4), n_init=20, random_state=42)
n_states_2d = selection_2d["bic"].idxmin()
model_2d = fit_gaussian_hmm(obs_2d, n_states=n_states_2d, n_init=30, random_state=42)

viterbi_2d = viterbi_decode(model_2d, obs_2d)
agreement_2d_vs_3d = (viterbi_2d == viterbi_states).mean() if n_states_2d == n_states_3d else None

print(f"2D seleciona {n_states_2d} estados (3D selecionou {n_states_3d})")
if agreement_2d_vs_3d is not None:
    print(f"Concordância Viterbi 2D vs. 3D: {agreement_2d_vs_3d:.1%}")
else:
    print("Nº de estados difere entre 2D e 3D — comparação direta de rótulo não é direta.")

characterize_regimes(model_2d, obs_2d, cols_2d)

Model is not converging.  Current: 23294.05178569435 is not greater than 23294.0614417995. Delta is -0.009656105146859773


Model is not converging.  Current: 21994.681434996473 is not greater than 22005.74878186606. Delta is -11.06734686958589


Model is not converging.  Current: 23294.05178569435 is not greater than 23294.0614417995. Delta is -0.009656105146859773


Model is not converging.  Current: 21994.681434996473 is not greater than 22005.74878186606. Delta is -11.06734686958589


Model is not converging.  Current: 23287.82216180246 is not greater than 23287.828431202688. Delta is -0.006269400226301514


Model is not converging.  Current: 23295.072183203254 is not greater than 23295.103298127175. Delta is -0.031114923920540605


2D seleciona 4 estados (3D selecionou 4)
Concordância Viterbi 2D vs. 3D: 58.9%


,persistence,expected_duration_days,mean_return_1d,mean_realized_vol_21d
state_rank,,,,
0,0.958903,24.332427,0.000471,0.128130
1,0.942869,17.503776,0.000451,0.168855
2,0.965441,28.936383,0.000723,0.226700
3,0.968843,32.095506,-0.003188,0.468714


## Salvar relatórios

In [7]:
selection_3d.to_csv(TABLES_DIR / "hmm_state_selection_3d.csv")
regime_summary_3d.to_csv(TABLES_DIR / "hmm_regime_characterization_3d.csv")

pd.Series(stability_3d).drop("selected_n_states_distribution").to_csv(TABLES_DIR / "hmm_bootstrap_stability_3d.csv")
stability_3d["selected_n_states_distribution"].to_csv(TABLES_DIR / "hmm_bootstrap_n_states_distribution_3d.csv")

print("Relatórios salvos em", TABLES_DIR)
print("Regimes (Viterbi + Forward) salvos em", PROCESSED_DIR / "hmm_regimes_insample.parquet")

Relatórios salvos em C:\Python\hmm_lstm_TCC\reports\tables
Regimes (Viterbi + Forward) salvos em C:\Python\hmm_lstm_TCC\data\processed\hmm_regimes_insample.parquet
